# 76장 · 같은 커밋을 검사하고 이미지 후보로 만들기

Pull Request의 테스트는 통과했는데 배포용 이미지가 생기지 않았습니다. 과정용 CI에서는 제안된 코드 검사와 main에 합쳐진 코드의 이미지 발행을 구분합니다. 한 workflow에서 이벤트, job, step이 각각 무엇을 정하는지 읽습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 PyYAML 6.0.3입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 두 작업이 실행되는 순서 읽기

step은 한 작업 안에서 차례로 진행합니다. job끼리 결과를 이어 쓰려면 needs로 의존 관계를 명시합니다. 아래는 실제 배포 양식에서 실행 조건을 뽑은 연습용 YAML입니다. 외부 저장소나 레지스트리를 변경하지 않습니다.



**볼 곳**　‘커밋’가 가운데 ‘검사 작업’에 도착한 뒤 오른쪽에 어떤 결과가 남는지 보세요.

**애니메이션의 의미**　같은 커밋의 검사가 통과해야 이미지 후보를 만듭니다.


### 셀 01 · 검사 다음 작업의 선행 조건 찾기

check는 코드를 검사하고 publish는 검사한 코드의 이미지 후보를 담당합니다. 어느 작업이 다른 작업을 기다리는지 보세요.

아래 YAML은 needs로 검사와 게시 순서를 정하고 if로 게시 조건을 좁히는 발췌입니다. 여기에는 저장소 checkout이나 실제 이미지 빌드·발행 단계가 없습니다. 커밋 문자열 출력과 이미지 발행 성공은 구별해 기록하세요.

In [1]:
import yaml
workflow = yaml.safe_load("""
name: textbook-ci
"on": [pull_request, push]
jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - name: 단위 검사
        run: python -m unittest
  publish:
    needs: check
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - name: 후보 식별
        run: echo "$GITHUB_SHA"
""")
print(workflow["jobs"]["publish"]["needs"])

check


check입니다. YAML에서 publish를 아래에 적었다는 사실만으로는 먼저 검사했다는 보장이 없습니다. needs가 작업의 연결을 만듭니다.

### 과제 02 · PR에서는 이미지를 발행하지 않도록 조건 걸기

아래 함수에서 main 브랜치로 push했고 검사가 통과한 경우에만 True를 반환하세요.

In [2]:
# may_publish(event, ref, ok)를 정의하고 PR 사례를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def may_publish(event, ref, ok):
    return event == &#x27;push&#x27; and ref == &#x27;refs/heads/main&#x27; and ok

print(may_publish(&#x27;pull_request&#x27;, &#x27;refs/pull/7/merge&#x27;, True))</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>검사에 성공한 PR이어도 False입니다. 레지스트리 쓰기 권한을 검사 작업에 줄 이유가 없습니다.</p>
</details>

## 로컬에서 실패가 다음 작업을 막는지 확인하기

CI runner는 검사 프로그램의 종료 코드를 받습니다. 학생의 노트북에서 별도 Python 프로세스로 작은 검사를 돌리고, 실패한 결과에서 후보를 발행해도 될지 판단합니다.



**볼 곳**　‘종료 코드 1’가 가운데 ‘검사 실패’에 도착한 뒤 오른쪽에 어떤 결과가 남는지 보세요.

**애니메이션의 의미**　검사 작업이 실패하면 뒤 작업은 멈춥니다.


### 셀 03 · 실패한 검사 프로세스의 종료 코드

잘못된 기대값을 가진 단위 검사를 실행합니다. 자식 프로세스의 실패가 노트북 전체를 중단시키지 않도록 결과를 받습니다.

In [3]:
import subprocess, sys
test = """import unittest
class TestPort(unittest.TestCase):
    def test_container_port(self):
        self.assertEqual(5432, 55432)
unittest.main()
"""
run = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True)
print(run.returncode)
print("5432 != 55432" in run.stderr)

1
True


1과 True가 보입니다. 실행한 단위 검사가 호스트 포트와 컨테이너 포트를 혼동했습니다. 실제 workflow라면 실패한 검사와 연결된 후보 발행을 멈춰야 합니다. GitHub-hosted runner에서 실행한 기록은 아닙니다.

## 돌아보기

실제 환경에서는 검사한 commit, 빌드한 commit, 발행한 이미지 digest가 가리키는 후보를 한 기록에서 대조하세요. 로컬 검사만으로 GitHub에서 이미지가 발행됐다고 기록하지 않습니다.

### 검사에서 게시 후보까지



**볼 곳**　같은 커밋에서 검사 결과와 브랜치 조건에 따라 게시 단계가 열리는지 보세요.

**애니메이션의 의미**　검사를 통과하고 게시 조건에 맞는 커밋만 이미지 후보가 됩니다.
